## Model Training : DistilBERT

This model uses frozen DistilBERT as contextual embedding layer, feeding into the existing BiGRU + Attention + Numerical architecture.

### Dataset Loading

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import DistilBertTokenizer
from torch.utils.data import DataLoader, TensorDataset
from model_construction.model import FakeJobDetector
import math

# Load preprocessed data
combined_df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")

numeric_cols = [
    "telecommuting", "missing_count", "total_text_len", "company_profile_len",
    "description_len", "requirements_len", "benefits_len",
    "company_profile_word_count", "description_word_count",
    "requirements_word_count", "benefits_word_count",
    "salary_provided", "has_company_profile", "vague_location",
    "has_company_logo", "has_questions"
]

print(f"Dataset: {combined_df.shape[0]} rows, {combined_df.shape[1]} cols")
print(f"Class balance: {combined_df['fraudulent'].value_counts().to_dict()}")

c:\Users\Chadrick\Documents\python_trial\fake-job-posting_DL_Project\.venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: 13485 rows, 18 cols
Class balance: {0: 12799, 1: 686}


### Train - Test - Validation Split

In [2]:
#Train / Val / Test split (80/10/10, stratified so same proportion of fraudulent vs non-fraudulent in each set)
train_data, temp_data = train_test_split(
    combined_df, test_size=0.2, random_state=42, stratify=combined_df['fraudulent']
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

X_train_text    = train_data['full_text'].tolist()
X_train_numeric = train_data[numeric_cols].values.astype(np.float32)
y_train         = train_data['fraudulent'].values.tolist()

X_val_text      = val_data['full_text'].tolist()
X_val_numeric   = val_data[numeric_cols].values.astype(np.float32)
y_val           = val_data['fraudulent'].values.tolist()

X_test_text     = test_data['full_text'].tolist()
X_test_numeric  = test_data[numeric_cols].values.astype(np.float32)
y_test          = test_data['fraudulent'].values.tolist()

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

Train: 10788, Val: 1348, Test: 1349


### Non-binary columns standardisation

In [3]:
#Scale non-binary numeric features
non_binary_cols = [
    "missing_count", "total_text_len", "company_profile_len", "description_len",
    "requirements_len", "benefits_len", "company_profile_word_count",
    "description_word_count", "requirements_word_count", "benefits_word_count"
]
non_binary_indices = [numeric_cols.index(col) for col in non_binary_cols]

scaler = StandardScaler()
X_train_numeric[:, non_binary_indices] = scaler.fit_transform(X_train_numeric[:, non_binary_indices])
X_val_numeric[:,   non_binary_indices] = scaler.transform(X_val_numeric[:,   non_binary_indices])
X_test_numeric[:,  non_binary_indices] = scaler.transform(X_test_numeric[:,  non_binary_indices])

### Tokenization with DistilBert

In [4]:
#Tokenize with DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
MAX_LEN = 512  # DistilBERT max sequence length

def tokenize_texts(texts, max_len):
    encoded = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    return encoded["input_ids"], encoded["attention_mask"]

train_input_ids, train_attention_masks = tokenize_texts(X_train_text, MAX_LEN)
val_input_ids, val_attention_masks     = tokenize_texts(X_val_text, MAX_LEN)
test_input_ids, test_attention_masks   = tokenize_texts(X_test_text, MAX_LEN)

print(f"Train tokens shape: {train_input_ids.shape}")  # (10788, 512)
print(f"Val tokens shape:   {val_input_ids.shape}")
print(f"Test tokens shape:  {test_input_ids.shape}")

Train tokens shape: torch.Size([10788, 512])
Val tokens shape:   torch.Size([1348, 512])
Test tokens shape:  torch.Size([1349, 512])


### Instantiate Datasets and DataLoaders

In [5]:
# Create Datasets and DataLoaders
BATCH_SIZE = 16  # smaller batch size since DistilBERT uses more memory
torch.manual_seed(42)
torch.cuda.manual_seed(42)

def make_dataset(input_ids, attention_masks, numeric, labels):
    return TensorDataset(
        input_ids,
        attention_masks,
        torch.tensor(numeric, dtype=torch.float),
        torch.tensor(labels, dtype=torch.float),
    )

train_dataset = make_dataset(train_input_ids, train_attention_masks, X_train_numeric, y_train)
val_dataset   = make_dataset(val_input_ids,   val_attention_masks,   X_val_numeric,   y_val)
test_dataset  = make_dataset(test_input_ids,  test_attention_masks,  X_test_numeric,  y_test)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_dataloader)}, Val: {len(val_dataloader)}, Test: {len(test_dataloader)}")

Train batches: 675, Val: 85, Test: 85


### Instantiating model 

In [6]:
# Instantiate model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FakeJobDetector(
    gru_hidden_dim         = 64,
    num_numerical_features = X_train_numeric.shape[1],
    num_hidden_dim         = 32,
    device                 = device,
)
print(f"Device: {device}")
print(model)

num_real = sum(1 for y in y_train if y == 0)
num_fake = sum(1 for y in y_train if y == 1)
print(f"Num real: {num_real}, Num fake: {num_fake}")

c:\Users\Chadrick\Documents\python_trial\fake-job-posting_DL_Project\.venv310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Chadrick\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8463.08it

Device: cuda
FakeJobDetector(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
       

### Model Training

In [ ]:
#Train
train_losses, val_losses = model.fit(
    dataloader     = train_dataloader,
    val_dataloader = val_dataloader,
    num_epochs     = 20,
    learning_rate  = 1e-3,
    save_path      = "best_model_distilbert.pt",
    pos_weight     = 3.0,
)

Epoch 1/20 | Train Loss: 0.3206 | Val Loss: 0.2485
  ✅ Best model saved (val_loss=0.2485)
Epoch 2/20 | Train Loss: 0.2188 | Val Loss: 0.1886
  ✅ Best model saved (val_loss=0.1886)
Epoch 3/20 | Train Loss: 0.1782 | Val Loss: 0.2048
  ⚠️ No improvement (best_val_loss=0.1886)
Epoch 4/20 | Train Loss: 0.1672 | Val Loss: 0.1746
  ✅ Best model saved (val_loss=0.1746)
Epoch 5/20 | Train Loss: 0.1553 | Val Loss: 0.1481
  ✅ Best model saved (val_loss=0.1481)
Epoch 6/20 | Train Loss: 0.1428 | Val Loss: 0.1435
  ✅ Best model saved (val_loss=0.1435)
Epoch 7/20 | Train Loss: 0.1358 | Val Loss: 0.2252
  ⚠️ No improvement (best_val_loss=0.1435)
Epoch 8/20 | Train Loss: 0.1208 | Val Loss: 0.1621
  ⚠️ No improvement (best_val_loss=0.1435)
Epoch 9/20 | Train Loss: 0.1240 | Val Loss: 0.2118
  ⚠️ No improvement (best_val_loss=0.1435)
Epoch 10/20 | Train Loss: 0.1190 | Val Loss: 0.1993
  ⚠️ No improvement (best_val_loss=0.1435)
Epoch 11/20 | Train Loss: 0.1192 | Val Loss: 0.1543
  ⚠️ No improvement (best_v

### Hyperparameter threshold tuning 

In [8]:
#Evaluate on test set
for threshold in [0.2, 0.3, 0.4, 0.5]:
    print(f"\n--- Threshold: {threshold} ---")
    model.evaluate(test_dataloader, threshold=threshold)


--- Threshold: 0.2 ---
              precision    recall  f1-score   support

        Real       0.99      0.96      0.97      1280
        Fake       0.52      0.87      0.65        69

    accuracy                           0.95      1349
   macro avg       0.75      0.91      0.81      1349
weighted avg       0.97      0.95      0.96      1349


--- Threshold: 0.3 ---
              precision    recall  f1-score   support

        Real       0.99      0.97      0.98      1280
        Fake       0.57      0.84      0.68        69

    accuracy                           0.96      1349
   macro avg       0.78      0.90      0.83      1349
weighted avg       0.97      0.96      0.96      1349


--- Threshold: 0.4 ---
              precision    recall  f1-score   support

        Real       0.99      0.97      0.98      1280
        Fake       0.64      0.83      0.72        69

    accuracy                           0.97      1349
   macro avg       0.82      0.90      0.85      1349
we